In [ ]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask
import glob
import pickle
import os
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, time


In [60]:
# === 1. Load SPOT CSV with Datetime column ===
spot_df = pd.read_csv(
    "/home/newberry3/main/Data/NIFTY_2021-06-01_to_2025-08-31_IDX.csv",
    usecols=["Date", "Time", "Open", "High", "Low", "Close"]
)

# Combine Date & Time into a Datetime column
spot_df["Datetime"] = pd.to_datetime(
    spot_df["Date"].astype(str) + " " + spot_df["Time"].astype(str), 
    errors='coerce'
)
num_bad_spot = spot_df["Datetime"].isna().sum()
if num_bad_spot > 0:
    print(f"⚠️  Dropping {num_bad_spot} bad rows from spot_df due to unparseable Datetime.")
    spot_df = spot_df.dropna(subset=["Datetime"])

spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

print("spot_df preview:")
print(spot_df.head())

spot_df.to_parquet("spot_data.parquet", index=False, engine="pyarrow")

spot_df preview:
             Datetime      Open      High       Low     Close
0 2021-06-01 09:15:00  15630.30  15630.30  15594.00  15613.55
1 2021-06-01 09:16:00  15610.50  15618.20  15607.60  15609.45
2 2021-06-01 09:17:00  15610.45  15611.10  15591.90  15599.85
3 2021-06-01 09:18:00  15598.75  15610.05  15588.65  15606.85
4 2021-06-01 09:19:00  15604.80  15606.40  15597.45  15602.95


In [27]:
# === 1. Load SPOT CSV with Datetime column ===
spot_df = pd.read_csv(
    "/home/newberry3/main/Data/SENSEX_2023-06-01_to_2025-08-31_IDX.csv",
    usecols=["Date", "Time", "Open", "High", "Low", "Close"]
)

# Combine Date & Time into a Datetime column
spot_df["Datetime"] = pd.to_datetime(
    spot_df["Date"].astype(str) + " " + spot_df["Time"].astype(str), 
    errors='coerce'
)
num_bad_spot = spot_df["Datetime"].isna().sum()
if num_bad_spot > 0:
    print(f"⚠️  Dropping {num_bad_spot} bad rows from spot_df due to unparseable Datetime.")
    spot_df = spot_df.dropna(subset=["Datetime"])

spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

print("spot_df preview:")
print(spot_df.head())

spot_df.to_parquet("spot_data_SENSEX.parquet", index=False, engine="pyarrow")


spot_df preview:
             Datetime      Open      High       Low     Close
0 2023-08-07 09:09:00  65811.40  65811.40  65811.40  65811.40
1 2023-08-07 09:15:00  65808.94  65872.49  65780.18  65865.27
2 2023-08-07 09:16:00  65860.06  65931.68  65849.22  65914.46
3 2023-08-07 09:17:00  65919.74  65919.74  65841.65  65854.68
4 2023-08-07 09:18:00  65853.58  65895.32  65848.06  65892.54


In [4]:
import os, glob, pickle, shutil
import dask, dask.dataframe as dd
import pandas as pd

# ---------- 1) Config ----------
options_files = glob.glob("/home/newberry3/main/Data/NIFTY/NIFTY_*.pkl")
if not options_files:
    raise FileNotFoundError("No NIFTY_*.pkl found")

STRING = pd.StringDtype()
target_cols = ["Date","Time","ExpiryDate","StrikePrice","Type","Open","High","Low","Close","Ticker"]
target_dtypes = {
    "Date": STRING, "Time": STRING, "ExpiryDate": STRING, "StrikePrice": STRING, "Type": STRING,
    "Open": "float64", "High": "float64", "Low": "float64", "Close": "float64", "Ticker": STRING,
}
meta = pd.DataFrame({c: pd.Series([], dtype=target_dtypes[c]) for c in target_cols})

# ---------- 2) Normalizer that DROPS OI/VOLUME before returning ----------
def load_and_normalize_pickle(path: str) -> pd.DataFrame:
    with open(path, "rb") as f:
        df = pickle.load(f)

    # Drop extras EARLY so from_delayed never sees them
    df = df.drop(columns=[c for c in ["OI", "Volume"] if c in df.columns], errors="ignore")

    # Ensure all required cols
    for c in target_cols:
        if c not in df.columns:
            df[c] = pd.NA

    # Cast text-ish
    for c in ["Date","Time","ExpiryDate","Type","Ticker"]:
        df[c] = df[c].astype("string")

    # StrikePrice: numeric -> Int64 -> string (avoid 18200.0)
    sp = pd.to_numeric(df["StrikePrice"], errors="coerce").astype("Int64")
    df["StrikePrice"] = sp.astype("string")

    # Prices -> float64
    for c in ["Open","High","Low","Close"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")

    # Return EXACT schema & order
    return df[target_cols]

# ---------- 3) Build Dask from delayed (all parts already normalized) ----------
delayed_parts = [dask.delayed(load_and_normalize_pickle)(p) for p in options_files]
options_df = dd.from_delayed(delayed_parts, meta=meta)

# Sanity: columns must match exactly (no OI)
assert list(options_df.columns) == target_cols, options_df.columns

# ---------- 4) Write Parquet ----------
shutil.rmtree("options_data.parquet", ignore_errors=True)
options_df.to_parquet("options_data.parquet", engine="pyarrow", write_index=False, compression="snappy")


/tmp/ipykernel_39502/2164807417.py:21: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  df = pickle.load(f)
/tmp/ipykernel_39502/2164807417.py:21: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use

In [40]:
import os, glob, pickle, shutil
import dask, dask.dataframe as dd
import pandas as pd

# ---------- 1) Config ----------
options_files = glob.glob("/home/newberry3/main/Data/SENSEX/new/SENSEX_*.pkl")
if not options_files:
    raise FileNotFoundError("No NIFTY_*.pkl found")

STRING = pd.StringDtype()
target_cols = ["Date","Time","ExpiryDate","StrikePrice","Type","Open","High","Low","Close","Ticker"]
target_dtypes = {
    "Date": STRING, "Time": STRING, "ExpiryDate": STRING, "StrikePrice": STRING, "Type": STRING,
    "Open": "float64", "High": "float64", "Low": "float64", "Close": "float64", "Ticker": STRING,
}
meta = pd.DataFrame({c: pd.Series([], dtype=target_dtypes[c]) for c in target_cols})

# ---------- 2) Normalizer that DROPS OI/VOLUME before returning ----------
def load_and_normalize_pickle(path: str) -> pd.DataFrame:
    with open(path, "rb") as f:
        df = pickle.load(f)

    # Drop extras EARLY so from_delayed never sees them
    df = df.drop(columns=[c for c in ["OI", "Volume"] if c in df.columns], errors="ignore")

    # Ensure all required cols
    for c in target_cols:
        if c not in df.columns:
            df[c] = pd.NA

    # Cast text-ish
    for c in ["Date","Time","ExpiryDate","Type","Ticker"]:
        df[c] = df[c].astype("string")

    # StrikePrice: numeric -> Int64 -> string (avoid 18200.0)
    sp = pd.to_numeric(df["StrikePrice"], errors="coerce").astype("Int64")
    df["StrikePrice"] = sp.astype("string")

    # Prices -> float64
    for c in ["Open","High","Low","Close"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")

    # Return EXACT schema & order
    return df[target_cols]

# ---------- 3) Build Dask from delayed (all parts already normalized) ----------
delayed_parts = [dask.delayed(load_and_normalize_pickle)(p) for p in options_files]
options_df = dd.from_delayed(delayed_parts, meta=meta)

# Sanity: columns must match exactly (no OI)
assert list(options_df.columns) == target_cols, options_df.columns

# ---------- 4) Write Parquet ----------
shutil.rmtree("options_data_SENSEX.parquet", ignore_errors=True)
options_df.to_parquet("options_data_SENSEX.parquet", engine="pyarrow", write_index=False, compression="snappy")


In [1]:
import pandas as pd
import numpy as np
from datetime import time

# ======================
# CONFIG
# ======================

# Entry times (HH:MM, 24h)
ENTRY_TIMES = [
    "09:25","09:50","10:15","10:25","10:50","11:25",
    "12:05","12:35","13:05","13:35","14:05"
]

# ATM rounding step per instrument
ATM_STEP = {
    "NIFTY": 50,
    "SENSEX": 100,
}

# Strike windows (points) per instrument and VIX bucket label
# Format: (ITM_points, OTM_points)
# Note: use 0 for ITM to indicate ATM (exact ATM) where specified.
STRIKE_WINDOWS = {
    # --- existing ---
    ("NIFTY",  "0-12"):      (100, 250),
    ("NIFTY",  "12-14.5"):   (100, 350),
    ("SENSEX", "0-12"):      (200, 500),
    ("SENSEX", "12-14.5"):   (100, 700),

    # --- NEW buckets ---
    # 14.5–18 VIX
    ("NIFTY",  "14.5-18"):   (100, 350),
    ("SENSEX", "14.5-18"):   (100, 600),

    # 18+ VIX
    ("NIFTY",  "18+"):       (50, 800),
    ("SENSEX", "18+"):       (0, 900),   # 0 = ATM for ITM leg, per your note
}

# Fill your dates here (YYYY-MM-DD) for each instrument & VIX bucket
# Existing examples kept; NEW buckets populated from your lists
DATES_BY_BUCKET = {
    # --- existing examples you already had ---
    ("NIFTY", "0-12"): [
        "2023-08-24","2023-06-08","2023-09-14",
        "2024-04-10","2024-04-04","2024-04-25",
        "2025-07-10","2025-07-24","2025-08-07",
    ],
    ("SENSEX", "0-12"): [
        "2023-08-25","2023-10-06","2023-11-24",
        "2024-04-05","2024-04-26","2024-09-27",
        "2025-08-05","2025-08-19","2025-07-15",
    ],
    ("NIFTY", "12-14.5"): [
        "2023-04-13","2023-05-18","2023-12-07",
        "2024-03-28","2024-07-25","2024-10-03",
        "2025-03-20","2025-04-03","2025-07-03",
    ],
    ("SENSEX", "12-14.5"): [
        "2023-08-18","2023-09-29","2023-12-15",
        "2024-03-07","2024-08-02","2024-12-13",
        "2025-02-25","2025-03-18","2025-06-24",
    ],

    # --- NEW buckets with your date lists ---
    # NIFTY: VIX 14.5–18
    ("NIFTY", "14.5-18"): [
        "2024-02-29","2024-05-09","2024-08-08",
        "2025-01-23","2025-02-13","2025-04-24",
        "2025-05-15","2024-11-28","2025-01-02","2025-01-16",
    ],
    # NIFTY: VIX > 18
    ("NIFTY", "18+"): [
        "2024-05-16","2024-05-23","2024-05-30","2024-06-06",
        "2025-01-30","2025-05-08","2025-05-29",
        "2022-09-29","2022-09-22",
    ],

    # SENSEX: VIX 14.5–18
    ("SENSEX", "14.5-18"): [
        "2024-03-01","2024-06-07","2024-07-19","2024-08-09","2024-08-16",
        "2024-11-01","2024-12-20","2025-01-07","2025-01-14","2025-01-21",
        "2025-02-18","2025-04-22",
    ],
    # SENSEX: VIX > 18
    ("SENSEX", "18+"): [
        "2025-01-28","2025-04-08","2025-04-15","2025-05-06","2025-05-13",
        "2024-05-10","2024-05-17","2024-05-24","2024-05-31","2025-05-27",
    ],
}

# Paths to your Parquet datasets
SPOT_PARQUET    = "spot_data.parquet"
OPTIONS_PARQUET = "options_data.parquet"


In [2]:
# ======================
# UTILITIES
# ======================

def to_date(s):
    return pd.to_datetime(s).date()

def round_to_step(x, step):
    return int(round(x / step) * step)

def nearest_weekly_expiry(expiry_series, trade_date):
    """Return the minimum ExpiryDate >= trade_date (as date) or NaT if none."""
    s = pd.to_datetime(expiry_series, errors="coerce").dt.date
    candidates = s[s >= trade_date]
    return candidates.min() if len(candidates) else pd.NaT

def build_strikes(atm, step, itm_pts, otm_pts):
    """
    Return all rounded strikes in a symmetric band around ATM using the *OTM* radius
    so both CE-OTM (above) and PE-OTM (below) are included.

    Example: ATM=18750, step=50, otm_pts=250 -> [18500, 18550, ..., 19000]
    """
    # Use OTM distance on both sides to cover CE/PE properly
    lo = round_to_step(atm - otm_pts, step)
    hi = round_to_step(atm + otm_pts, step)

    # Safety swap
    if lo > hi:
        lo, hi = hi, lo

    # Inclusive list of strikes
    return list(range(int(lo), int(hi) + int(step), int(step)))




In [3]:
# ======================
# LOAD DATA
# ======================

# Spot minute data: Datetime, Open, High, Low, Close
spot_df = pd.read_parquet(SPOT_PARQUET)
spot_df["Datetime"] = pd.to_datetime(spot_df["Datetime"], errors="coerce")
spot_df["Date"] = spot_df["Datetime"].dt.date
spot_df["Time"] = spot_df["Datetime"].dt.strftime("%H:%M")

# We’ll use Close as spot for ATM computation at entry times
spot_cols = ["Date","Time","Close"]
spot_df_n = spot_df[spot_cols].rename(columns={"Close": "Spot"}).copy()

# Options minute data
# Columns: Date, Time, ExpiryDate, StrikePrice, Type(CE/PE), Open, High, Low, Close, Ticker
opt_df = pd.read_parquet(OPTIONS_PARQUET).copy()
# Normalize types
opt_df["Date"] = pd.to_datetime(opt_df["Date"], errors="coerce").dt.date
opt_df["Time"] = pd.to_datetime(opt_df["Time"], format="%H:%M", errors="coerce").dt.strftime("%H:%M")
opt_df["ExpiryDate"] = pd.to_datetime(opt_df["ExpiryDate"], errors="coerce").dt.date



In [4]:
# ======================
# CORE EXTRACTION
# ======================

records = []

for (instrument, vix_bucket), date_list in DATES_BY_BUCKET.items():
    if not date_list:
        continue

    step = ATM_STEP[instrument]
    itm_pts, otm_pts = STRIKE_WINDOWS[(instrument, vix_bucket)]

    # Subset options for speed: just those dates
    dates_set = set(pd.to_datetime(date_list).date)
    opt_sub = opt_df[opt_df["Date"].isin(dates_set)].copy()

    # If your options parquet includes multiple tickers (e.g., NIFTY/SENSEX), filter by Ticker prefix:
    # Expecting Ticker like "2023090109:15NIFTY23090718200PE" — we can detect by substring.
    opt_sub = opt_sub[opt_sub["Ticker"].astype(str).str.contains(instrument, na=False)]

    # For each date, determine nearest expiry for that day
    # (Among rows of that date, pick min ExpiryDate >= date)
    for d in sorted(dates_set):
        day_rows = opt_sub[opt_sub["Date"] == d]
        if day_rows.empty:
            continue
        expiry = nearest_weekly_expiry(day_rows["ExpiryDate"], d)
        if pd.isna(expiry):
            continue

        # Keep only that expiry for the day
        day_rows = day_rows[day_rows["ExpiryDate"] == expiry]

        # Build a little lookup for price window queries later
        # We’ll also need spot at the entry times to get ATM
        day_spot = spot_df_n[spot_df_n["Date"] == d]
        if day_spot.empty:
            continue

        # For each entry time:
        for t in ENTRY_TIMES:
            # Spot at entry time t (first value at or after t)
            srow = day_spot[day_spot["Time"] >= t].head(1)
            if srow.empty:
                # fallback: nearest before
                srow = day_spot[day_spot["Time"] <= t].tail(1)
            if srow.empty:
                continue

            spot = float(srow["Spot"].iloc[0])
            atm = round_to_step(spot, step)
            strikes = build_strikes(atm, step, itm_pts, otm_pts)

            # Entry window is exactly the minute 't'; window end is 15:15
            window_end = "15:15"

            # Filter rows from t..end for the day/expiry, then within strikes
            mask_window = (day_rows["Time"] >= t) & (day_rows["Time"] <= window_end)
            day_win = day_rows[mask_window]
            if day_win.empty:
                continue

            day_entry = day_rows[day_rows["Time"] == t]
            if day_entry.empty:
                # If exact minute missing, use first minute *after* t as entry
                day_entry = day_rows[day_rows["Time"] > t].head(1)
                if day_entry.empty:
                    # else use last before t
                    day_entry = day_rows[day_rows["Time"] < t].tail(1)
                    if day_entry.empty:
                        continue

            # Work only the requested strikes (rounded)
            day_entry = day_entry[day_entry["StrikePrice"].astype(int).isin(strikes)]
            day_win   = day_win[day_win["StrikePrice"].astype(int).isin(strikes)]
            if day_entry.empty or day_win.empty:
                continue

            # For each Type and each Strike, compute entry O/C and window High/Low
            # Entry O/C taken from that minute row (if multiple ticks, take first)
            grp_entry = (
                day_entry
                .sort_values(["StrikePrice","Type"])
                .groupby(["StrikePrice","Type"], as_index=False)
                .agg(Entry_Open=("Open","first"), Entry_Close=("Close","first"))
            )

            grp_window = (
                day_win
                .groupby(["StrikePrice","Type"], as_index=False)
                .agg(Window_High=("High","max"), Window_Low=("Low","min"))
            )

            out = pd.merge(grp_entry, grp_window, on=["StrikePrice","Type"], how="inner")
            if out.empty:
                continue

            out.insert(0, "Instrument", instrument)
            out.insert(1, "VIX_Bucket", vix_bucket)
            out.insert(2, "Date", pd.to_datetime(d))
            out.insert(3, "Entry_Time", t)
            out.insert(4, "ExpiryDate", pd.to_datetime(expiry))

            # Optional: add ATM and Spot-at-entry for context
            out["ATM"]  = atm
            out["Spot"] = spot

            records.append(out)


In [7]:
# ======================
# OUTPUT
# ======================

if records:
    final_df = pd.concat(records, ignore_index=True)
    # Sort nicely
    final_df = final_df.sort_values(
        ["Instrument","VIX_Bucket","Date","Entry_Time","StrikePrice","Type"]
    ).reset_index(drop=True)

    final_df.to_csv("nifty_premium_windows_by_bucket.csv", index=False)
    print(final_df.head(20))
    print(f"\nSaved {len(final_df)} rows to premium_windows_by_bucket.csv")
else:
    print("No rows produced. Check dates, parquet paths, and Ticker filtering.")


   Instrument VIX_Bucket       Date Entry_Time ExpiryDate StrikePrice Type  \
0       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18500   CE   
1       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18500   PE   
2       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18550   CE   
3       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18550   PE   
4       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18600   CE   
5       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18600   PE   
6       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18650   CE   
7       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18650   PE   
8       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18700   CE   
9       NIFTY       0-12 2023-06-08      09:25 2023-06-08       18700   PE   
10      NIFTY       0-12 2023-06-08      09:25 2023-06-08       18750   CE   
11      NIFTY       0-12 2023-06-08      09:25 2023-06-08       

In [ ]:
# import os
# import pandas as pd
# from pathlib import Path

# # ===== CONFIG =====
# SRC = "/home/newberry3/main/Data/nifty_premium_windows_by_bucket.csv"
# OUT_DIR = "/home/newberry3/main/Data/nifty_premium_windows_excel"  # output folder
# EXCEL_ENGINE = "openpyxl"  # use openpyxl instead of xlsxwriter

# # ===== LOAD =====
# df = pd.read_csv(SRC)

# # Create output dir
# Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# # Helper: Excel-safe sheet names
# def sheet_name_from_time(t: str) -> str:
#     s = f"T_{str(t)}".replace(":", "_")
#     for bad in [":", "\\", "/", "?", "*", "[", "]"]:
#         s = s.replace(bad, "_")
#     return s[:31]

# # ===== BUILD EXCELS =====
# for bucket, df_bucket in df.groupby("VIX_Bucket", dropna=False):
#     bucket_str = str(bucket).replace("/", "-")
#     xlsx_path = os.path.join(OUT_DIR, f"nifty_premium_{bucket_str}.xlsx")

#     # Sort rows so sheets appear in time order
#     df_bucket = df_bucket.sort_values(
#         ["Entry_Time", "Instrument", "Date", "ExpiryDate", "StrikePrice", "Type"]
#     )

#     with pd.ExcelWriter(xlsx_path, engine=EXCEL_ENGINE) as writer:
#         for entry_time, df_time in df_bucket.groupby("Entry_Time", dropna=False):
#             sheet = sheet_name_from_time(entry_time)
#             df_time.to_excel(writer, sheet_name=sheet, index=False)

#     print(f"✅ Wrote {xlsx_path}")


✅ Wrote /home/newberry3/main/Data/nifty_premium_windows_excel/nifty_premium_0-12.xlsx
✅ Wrote /home/newberry3/main/Data/nifty_premium_windows_excel/nifty_premium_12-14.5.xlsx


In [6]:
# #readjusted columns final
# import os
# import pandas as pd
# from pathlib import Path

# # ===== CONFIG =====
# SRC = "/home/newberry3/main/Data/nifty_premium_windows_by_bucket.csv"
# OUT_DIR = "/home/newberry3/main/Data/nifty_premium_windows_excel_flat"
# ENGINE = "openpyxl"         # or "xlsxwriter" if installed
# PRICE_FIELD = "Entry_Open"  # or "Entry_Open"

# ATM_STEP = {"NIFTY": 50, "SENSEX": 100}

# # Strike windows (points) per instrument & VIX bucket
# STRIKE_WINDOWS = {
#     ("NIFTY",  "0-12"):    (100, 250),
#     ("NIFTY",  "12-14.5"): (100, 350),
#     ("SENSEX", "0-12"):    (200, 500),
#     ("SENSEX", "12-14.5"): (100, 700),
# }

# # ===== LOAD =====
# df = pd.read_csv(SRC)

# need = {
#     "Instrument","VIX_Bucket","Date","Entry_Time","ExpiryDate","StrikePrice","Type",
#     "ATM","Spot", PRICE_FIELD, "Window_High", "Window_Low"
# }
# miss = need - set(df.columns)
# if miss:
#     raise ValueError(f"Missing columns in input: {miss}")

# # Normalize numeric
# df["StrikePrice"] = pd.to_numeric(df["StrikePrice"], errors="coerce").astype("Int64")
# df["ATM"]         = pd.to_numeric(df["ATM"], errors="coerce").astype("Int64")
# df["Spot"]        = pd.to_numeric(df["Spot"], errors="coerce")

# Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# def sheet_name_from_time(t: str) -> str:
#     s = f"T_{str(t)}".replace(":", "_")
#     for bad in [":","\\","/","?","*","[","]"]:
#         s = s.replace(bad, "_")
#     return s[:31]

# def build_flat_rows(df_slice: pd.DataFrame, vix_bucket: str) -> pd.DataFrame:
#     """
#     For a subset fixed to one Entry_Time (but possibly multiple dates),
#     build a flattened table using STRIKE_WINDOWS for the instrument/bucket,
#     including Entry price and Window High/Low for each strike × {CE,PE}.
#     """
#     rows = []

#     # Determine per-instrument step and max ITM/OTM depth (in #steps)
#     max_cols_spec = {}
#     for inst in df_slice["Instrument"].dropna().unique():
#         inst_u = str(inst).upper()
#         step = ATM_STEP.get(inst_u, 50)
#         itm_pts, otm_pts = STRIKE_WINDOWS[(inst_u, vix_bucket)]
#         max_cols_spec[inst] = (step, itm_pts // step, otm_pts // step)

#     # Build rows per (Instrument, Date, ExpiryDate)
#     for (inst, date, expiry), g in df_slice.groupby(["Instrument","Date","ExpiryDate"], dropna=False):
#         if pd.isna(inst) or pd.isna(expiry):
#             continue
#         step, k_itm_max, k_otm_max = max_cols_spec[inst]
#         atm_series = g["ATM"].dropna().astype("Int64")
#         if atm_series.empty:
#             continue
#         atm = int(atm_series.iloc[0])
#         spot = g["Spot"].dropna().iloc[0] if not g["Spot"].dropna().empty else pd.NA
#         entry_time = g["Entry_Time"].iloc[0]

#         # One row lookup table by (StrikePrice, Type) -> values
#         # Use first occurrence (should be unique per minute anyway)
#         key_cols = ["StrikePrice","Type"]
#         val_cols = [PRICE_FIELD, "Window_High", "Window_Low"]
#         grp = (g.sort_values(key_cols)
#                  .drop_duplicates(subset=key_cols, keep="first")
#                  .set_index(key_cols)[val_cols])

#         def val_dict(strike: int, typ: str):
#             """Return dict with {price, high, low} for given (strike, typ)."""
#             try:
#                 rec = grp.loc[(int(strike), typ)]
#                 return {
#                     "price": rec[PRICE_FIELD],
#                     "high":  rec["Window_High"],
#                     "low":   rec["Window_Low"],
#                 }
#             except Exception:
#                 return {"price": pd.NA, "high": pd.NA, "low": pd.NA}

#         # Base row
#         row = {
#             "date": date,
#             "time": entry_time,
#             "index": inst,
#             "expiry": expiry,
#             "spot": spot,
#             "atm strike": atm,
#         }

#         # ATM CE/PE
#         atm_ce = val_dict(atm, "CE")
#         atm_pe = val_dict(atm, "PE")
#         row.update({
#             "atm ce": atm_ce["price"],
#             "atm ce high": atm_ce["high"],
#             "atm ce low":  atm_ce["low"],
#             "atm pe": atm_pe["price"],
#             "atm pe high": atm_pe["high"],
#             "atm pe low":  atm_pe["low"],
#         })

#         # ITM ladder
#         for k in range(1, k_itm_max + 1):
#             s_itm = atm - k * step
#             ce = val_dict(s_itm, "CE")
#             pe = val_dict(s_itm, "PE")
#             row[f"itm{k} strike"]    = s_itm
#             row[f"itm{k} ce"]        = ce["price"]
#             row[f"itm{k} ce high"]   = ce["high"]
#             row[f"itm{k} ce low"]    = ce["low"]
#             row[f"itm{k} pe"]        = pe["price"]
#             row[f"itm{k} pe high"]   = pe["high"]
#             row[f"itm{k} pe low"]    = pe["low"]

#         # OTM ladder
#         for k in range(1, k_otm_max + 1):
#             s_otm = atm + k * step
#             ce = val_dict(s_otm, "CE")
#             pe = val_dict(s_otm, "PE")
#             row[f"otm{k} strike"]    = s_otm
#             row[f"otm{k} ce"]        = ce["price"]
#             row[f"otm{k} ce high"]   = ce["high"]
#             row[f"otm{k} ce low"]    = ce["low"]
#             row[f"otm{k} pe"]        = pe["price"]
#             row[f"otm{k} pe high"]   = pe["high"]
#             row[f"otm{k} pe low"]    = pe["low"]

#         rows.append(row)

#     if not rows:
#         return pd.DataFrame()

#     # Build consistent column order for the sheet
#     k_itm_max_sheet = 0
#     k_otm_max_sheet = 0
#     for inst in df_slice["Instrument"].dropna().unique():
#         _, itm_max, otm_max = max_cols_spec[inst]
#         k_itm_max_sheet = max(k_itm_max_sheet, itm_max)
#         k_otm_max_sheet = max(k_otm_max_sheet, otm_max)

#     cols = [
#         "date","time","index","expiry","spot",
#         "atm strike","atm ce","atm ce high","atm ce low","atm pe","atm pe high","atm pe low",
#     ]
#     for k in range(1, k_itm_max_sheet + 1):
#         cols += [
#             f"itm{k} strike",
#             f"itm{k} ce",      f"itm{k} ce high",  f"itm{k} ce low",
#             f"itm{k} pe",      f"itm{k} pe high",  f"itm{k} pe low",
#         ]
#     for k in range(1, k_otm_max_sheet + 1):
#         cols += [
#             f"otm{k} strike",
#             f"otm{k} ce",      f"otm{k} ce high",  f"otm{k} ce low",
#             f"otm{k} pe",      f"otm{k} pe high",  f"otm{k} pe low",
#         ]

#     out = pd.DataFrame(rows)
#     # Ensure missing columns exist (rectangular sheet)
#     for c in cols:
#         if c not in out.columns:
#             out[c] = pd.NA

#     return out[cols].sort_values(["date","time","index","expiry"]).reset_index(drop=True)

# # ===== WRITE EXCELS =====
# for bucket, df_bucket in df.groupby("VIX_Bucket", dropna=False):
#     bucket_str = str(bucket).replace("/", "-")
#     xlsx_path = os.path.join(OUT_DIR, f"nifty_flat_{bucket_str}.xlsx")

#     with pd.ExcelWriter(xlsx_path, engine=ENGINE) as writer:
#         # Sheet per Entry_Time (chronological)
#         for entry_time, df_time in df_bucket.sort_values("Entry_Time").groupby("Entry_Time", dropna=False):
#             sheet = sheet_name_from_time(entry_time)
#             flat = build_flat_rows(df_time, vix_bucket=bucket_str)
#             if flat.empty:
#                 continue
#             flat.to_excel(writer, sheet_name=sheet, index=False)

#     print(f"✅ Wrote {xlsx_path}")


In [8]:
#readjusted columns final
import os
import pandas as pd
from pathlib import Path

# ===== CONFIG =====
SRC = "/home/newberry3/main/Data/nifty_premium_windows_by_bucket.csv"
OUT_DIR = "/home/newberry3/main/Data/nifty_premium_windows_excel_flat"
ENGINE = "openpyxl"         # or "xlsxwriter" if installed
PRICE_FIELD = "Entry_Open"  # or "Entry_Open"

ATM_STEP = {"NIFTY": 50, "SENSEX": 100}

# Strike windows (points) per instrument & VIX bucket
STRIKE_WINDOWS = {
    ("NIFTY",  "0-12"):      (100, 250),
    ("NIFTY",  "12-14.5"):   (100, 350),
    ("NIFTY",  "14.5-18"):   (100, 350),
    ("NIFTY",  "18+"):       (50,  800),

    ("SENSEX", "0-12"):      (200, 500),
    ("SENSEX", "12-14.5"):   (100, 700),
    ("SENSEX", "14.5-18"):   (100, 600),
    ("SENSEX", "18+"):       (0,   900),  # 0 => ITM leg is ATM
}

# ===== LOAD =====
df = pd.read_csv(SRC)

need = {
    "Instrument","VIX_Bucket","Date","Entry_Time","ExpiryDate","StrikePrice","Type",
    "ATM","Spot", PRICE_FIELD, "Window_High", "Window_Low"
}
miss = need - set(df.columns)
if miss:
    raise ValueError(f"Missing columns in input: {miss}")

# Normalize numeric
df["StrikePrice"] = pd.to_numeric(df["StrikePrice"], errors="coerce").astype("Int64")
df["ATM"]         = pd.to_numeric(df["ATM"], errors="coerce").astype("Int64")
df["Spot"]        = pd.to_numeric(df["Spot"], errors="coerce")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def sheet_name_from_time(t: str) -> str:
    s = f"T_{str(t)}".replace(":", "_")
    for bad in [":","\\","/","?","*","[","]"]:
        s = s.replace(bad, "_")
    return s[:31]

def build_flat_rows(df_slice: pd.DataFrame, vix_bucket: str) -> pd.DataFrame:
    """
    For a subset fixed to one Entry_Time (but possibly multiple dates),
    build a flattened table using STRIKE_WINDOWS for the instrument/bucket,
    including Entry price and Window High/Low for each strike × {CE,PE}.
    """
    rows = []

    # Determine per-instrument step and max ITM/OTM depth (in #steps)
    max_cols_spec = {}
    for inst in df_slice["Instrument"].dropna().unique():
        inst_u = str(inst).upper()
        step = ATM_STEP.get(inst_u, 50)
        itm_pts, otm_pts = STRIKE_WINDOWS[(inst_u, vix_bucket)]
        max_cols_spec[inst] = (step, itm_pts // step, otm_pts // step)

    # Build rows per (Instrument, Date, ExpiryDate)
    for (inst, date, expiry), g in df_slice.groupby(["Instrument","Date","ExpiryDate"], dropna=False):
        if pd.isna(inst) or pd.isna(expiry):
            continue
        step, k_itm_max, k_otm_max = max_cols_spec[inst]
        atm_series = g["ATM"].dropna().astype("Int64")
        if atm_series.empty:
            continue
        atm = int(atm_series.iloc[0])
        spot = g["Spot"].dropna().iloc[0] if not g["Spot"].dropna().empty else pd.NA
        entry_time = g["Entry_Time"].iloc[0]

        # One row lookup table by (StrikePrice, Type) -> values
        # Use first occurrence (should be unique per minute anyway)
        key_cols = ["StrikePrice","Type"]
        val_cols = [PRICE_FIELD, "Window_High", "Window_Low"]
        grp = (g.sort_values(key_cols)
                 .drop_duplicates(subset=key_cols, keep="first")
                 .set_index(key_cols)[val_cols])

        def val_dict(strike: int, typ: str):
            """Return dict with {price, high, low} for given (strike, typ)."""
            try:
                rec = grp.loc[(int(strike), typ)]
                return {
                    "price": rec[PRICE_FIELD],
                    "high":  rec["Window_High"],
                    "low":   rec["Window_Low"],
                }
            except Exception:
                return {"price": pd.NA, "high": pd.NA, "low": pd.NA}

        # Base row
        row = {
            "date": date,
            "time": entry_time,
            "index": inst,
            "expiry": expiry,
            "spot": spot,
            "atm strike": atm,
        }

        # ATM CE/PE
        atm_ce = val_dict(atm, "CE")
        atm_pe = val_dict(atm, "PE")
        row.update({
            "atm ce": atm_ce["price"],
            "atm ce high": atm_ce["high"],
            "atm ce low":  atm_ce["low"],
            "atm pe": atm_pe["price"],
            "atm pe high": atm_pe["high"],
            "atm pe low":  atm_pe["low"],
        })

        # ======== CHANGED: type-aware ITM/OTM labeling =========
        # CE-ITM = below ATM; PE-ITM = above ATM
        for k in range(1, k_itm_max + 1):
            ce_itm_strike = atm - k * step
            pe_itm_strike = atm + k * step

            ce = val_dict(ce_itm_strike, "CE")
            pe = val_dict(pe_itm_strike, "PE")

            row[f"ce_itm{k} strike"] = ce_itm_strike
            row[f"ce_itm{k}"]        = ce["price"]
            row[f"ce_itm{k} high"]   = ce["high"]
            row[f"ce_itm{k} low"]    = ce["low"]

            row[f"pe_itm{k} strike"] = pe_itm_strike
            row[f"pe_itm{k}"]        = pe["price"]
            row[f"pe_itm{k} high"]   = pe["high"]
            row[f"pe_itm{k} low"]    = pe["low"]

        # CE-OTM = above ATM; PE-OTM = below ATM
        for k in range(1, k_otm_max + 1):
            ce_otm_strike = atm + k * step
            pe_otm_strike = atm - k * step

            ce = val_dict(ce_otm_strike, "CE")
            pe = val_dict(pe_otm_strike, "PE")

            row[f"ce_otm{k} strike"] = ce_otm_strike
            row[f"ce_otm{k}"]        = ce["price"]
            row[f"ce_otm{k} high"]   = ce["high"]
            row[f"ce_otm{k} low"]    = ce["low"]

            row[f"pe_otm{k} strike"] = pe_otm_strike
            row[f"pe_otm{k}"]        = pe["price"]
            row[f"pe_otm{k} high"]   = pe["high"]
            row[f"pe_otm{k} low"]    = pe["low"]
        # ======== /CHANGED =========

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    # Build consistent column order for the sheet
    k_itm_max_sheet = 0
    k_otm_max_sheet = 0
    for inst in df_slice["Instrument"].dropna().unique():
        _, itm_max, otm_max = max_cols_spec[inst]
        k_itm_max_sheet = max(k_itm_max_sheet, itm_max)
        k_otm_max_sheet = max(k_otm_max_sheet, otm_max)

    cols = [
        "date","time","index","expiry","spot",
        "atm strike","atm ce","atm ce high","atm ce low","atm pe","atm pe high","atm pe low",
    ]
    # ======== CHANGED: new column order for type-aware ITM/OTM =========
    for k in range(1, k_itm_max_sheet + 1):
        cols += [
            f"ce_itm{k} strike", f"ce_itm{k}", f"ce_itm{k} high", f"ce_itm{k} low",
            f"pe_itm{k} strike", f"pe_itm{k}", f"pe_itm{k} high", f"pe_itm{k} low",
        ]
    for k in range(1, k_otm_max_sheet + 1):
        cols += [
            f"ce_otm{k} strike", f"ce_otm{k}", f"ce_otm{k} high", f"ce_otm{k} low",
            f"pe_otm{k} strike", f"pe_otm{k}", f"pe_otm{k} high", f"pe_otm{k} low",
        ]
    # ======== /CHANGED =========

    out = pd.DataFrame(rows)
    # Ensure missing columns exist (rectangular sheet)
    for c in cols:
        if c not in out.columns:
            out[c] = pd.NA

    return out[cols].sort_values(["date","time","index","expiry"]).reset_index(drop=True)

# ===== WRITE EXCELS =====
for bucket, df_bucket in df.groupby("VIX_Bucket", dropna=False):
    bucket_str = str(bucket).replace("/", "-")
    xlsx_path = os.path.join(OUT_DIR, f"nifty_flat_{bucket_str}.xlsx")

    with pd.ExcelWriter(xlsx_path, engine=ENGINE) as writer:
        # Sheet per Entry_Time (chronological)
        for entry_time, df_time in df_bucket.sort_values("Entry_Time").groupby("Entry_Time", dropna=False):
            sheet = sheet_name_from_time(entry_time)
            flat = build_flat_rows(df_time, vix_bucket=bucket_str)
            if flat.empty:
                continue
            flat.to_excel(writer, sheet_name=sheet, index=False)

    print(f"✅ Wrote {xlsx_path}")


✅ Wrote /home/newberry3/main/Data/nifty_premium_windows_excel_flat/nifty_flat_0-12.xlsx
✅ Wrote /home/newberry3/main/Data/nifty_premium_windows_excel_flat/nifty_flat_12-14.5.xlsx
✅ Wrote /home/newberry3/main/Data/nifty_premium_windows_excel_flat/nifty_flat_14.5-18.xlsx
✅ Wrote /home/newberry3/main/Data/nifty_premium_windows_excel_flat/nifty_flat_18+.xlsx
